# Cognito — Sistema de Inferência com Paging Dinâmico de KV Cache em Nível de Aplicação

**Trabalho de Conclusão de Curso — Anexo Técnico de Implementação**

Este notebook documenta a implementação de referência do sistema Cognito. O pipeline é organizado em três fases executadas em ambientes isolados:

| Fase | Script | Descrição |
|------|--------|-----------|
| 1 | `1_ingestao.py` | Extração do corpus TriviaQA e indexação vetorial via ChromaDB |
| 2 | `2_avaliacao.py` | Benchmarking padronizado (ARC, HellaSwag, WinoGrande, MMLU, TriviaQA) via LM Evaluation Harness |
| 3 | `3_inferencia.py` | Pipeline RAG completo com `VirtualPageManager` e avaliação comparativa de pico de VRAM |

**Ambiente de referência:** Google Colab — GPU NVIDIA T4 (16 GB VRAM)  
**Modelo:** `mistralai/Mistral-7B-Instruct-v0.3` — quantização NF4 (bitsandbytes)  
**Bibliotecas:** HuggingFace Transformers ≥ 4.46, bitsandbytes, ChromaDB, sentence-transformers, lm-eval

---

**Justificativa Arquitetural — `%%writefile` e Isolamento de Kernel:**  
Cada fase é escrita via `%%writefile` e executada como script autônomo pelo gerenciador `uv` (`!uv run script.py`). Essa decisão garante dois requisitos científicos críticos:

1. **Isolamento de Estado (VRAM):** O interpretador IPython retém tensores na GPU entre células. A execução como processo separado garante que a VRAM seja integralmente liberada ao término de cada fase.
2. **Reproducibilidade de Dependências:** Cada fase declara suas dependências via PEP 723 (inline script metadata), permitindo que o `uv` instale um ambiente exato e isolado — eliminando conflitos entre bibliotecas como `bitsandbytes`, `lm-eval` e `torchvision` presentes na imagem base do Colab.


## 0. Gestão de Sessão — Google Drive

Execute a **Célula de Início** ao abrir o Colab e a **Célula de Fim** antes de fechar.

| O que persiste no Drive | Por quê |
|-------------------------|----------|
| `chroma_cognito/` | ChromaDB — caro de reconstruir (~10 min embedding) |
| `gold_answers.json` | Queries de avaliação e gabaritos |
| `results_checkpoint_C*.jsonl` | Resultados — perder esses é perder sessões de GPU |
| Modelo Mistral | **Não persiste** — HF Hub re-download (~1 min) é mais rápido que Drive |

> **Estratégia de I/O:** ChromaDB usa SQLite com leituras aleatórias intensas.  
> Rodar direto do Drive seria 3-5× mais lento. A cópia Drive → `/content/` leva ~15-30s e elimina o gargalo.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA DE INÍCIO DE SESSÃO — Execute SEMPRE ao abrir o Colab
# ═══════════════════════════════════════════════════════════════
# Monta o Drive e copia dados persistentes para o SSD local (/content/).
# ChromaDB tem I/O aleatório intenso — rodar direto do Drive seria
# 3-5x mais lento. A cópia local demora ~10-30s e resolve o problema.

import os, shutil, time
from google.colab import drive

drive.mount('/content/drive')

# ── Configuração de caminhos ─────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/cognito_tcc'
LOCAL_ROOT  = '/content'

# Cria diretórios no Drive se não existirem
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/chroma_cognito', exist_ok=True)

# ── Restaura ChromaDB (índice vetorial) ──────────────────────────
DRIVE_CHROMA = f'{DRIVE_ROOT}/chroma_cognito'
LOCAL_CHROMA = f'{LOCAL_ROOT}/chroma_cognito'
chroma_sqlite = f'{DRIVE_CHROMA}/chroma.sqlite3'

if os.path.exists(chroma_sqlite):
    print('Restaurando ChromaDB do Drive → /content/ ...')
    t0 = time.time()
    if os.path.exists(LOCAL_CHROMA):
        shutil.rmtree(LOCAL_CHROMA)
    shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA)
    size_mb = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(LOCAL_CHROMA) for f in fs) / 1e6
    print(f'  ✓ ChromaDB restaurado: {size_mb:.1f} MB em {time.time()-t0:.1f}s')
else:
    print('⚠ ChromaDB não encontrado no Drive.')
    print('  Execute a célula de ingestão (1_ingestao.py) para construí-lo.')

# ── Restaura gold_answers.json ────────────────────────────────────
DRIVE_GOLD = f'{DRIVE_ROOT}/gold_answers.json'
if os.path.exists(DRIVE_GOLD):
    shutil.copy(DRIVE_GOLD, f'{LOCAL_ROOT}/gold_answers.json')
    import json as _j
    n = len(_j.load(open(f'{LOCAL_ROOT}/gold_answers.json')))
    print(f'  ✓ gold_answers.json restaurado: {n} queries')
else:
    print('  ⚠ gold_answers.json não encontrado — será gerado pela ingestão.')

# ── Restaura checkpoints de runs anteriores ───────────────────────
DRIVE_CKPTS = f'{DRIVE_ROOT}/checkpoints'
ckpts = [f for f in os.listdir(DRIVE_CKPTS) if f.endswith('.jsonl')]
for ck in ckpts:
    shutil.copy(f'{DRIVE_CKPTS}/{ck}', f'{LOCAL_ROOT}/{ck}')
if ckpts:
    print(f'  ✓ Checkpoints restaurados: {ckpts}')
else:
    print('  ℹ Nenhum checkpoint anterior encontrado (primeira sessão).')

print()
print('✓ Sessão restaurada. Dados em /content/ prontos para uso.')
print(f'  Drive root: {DRIVE_ROOT}')


# TODO:
- [x] **Está consumado...**

## 1. Instalação de Dependências

In [ ]:
%%writefile 1_ingestao.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "datasets>=2.20.0",
#     "sentence-transformers>=3.0.0",
#     "chromadb>=0.5.5",
#     "torch",
#     "einops"
# ]
# ///

import json
import chromadb
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore")

EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
VECTOR_DB_PATH       = "./chroma_cognito"

# ── Separação Corpus / Avaliação (Fase 0 — Roadmap) ──────────────────────────
# Documentos indexados vêm do split TRAIN do TriviaQA.
# As queries de avaliação vêm do split VALIDATION (conjunto independente).
# Isso elimina data leakage: o modelo não pode "lembrar" contextos que estão
# no mesmo split das perguntas de teste.
CORPUS_SPLIT     = "train[:600]"      # Documentos para o índice vetorial
EVAL_SPLIT       = "validation[:200]" # Perguntas de avaliação (independentes)

MIN_PASSAGE_CHARS   = 500   # Mantém apenas passagens substantivas
MAX_CORPUS_PASSAGES = 1000  # Limite superior do corpus indexado


def main():
    # ── Corpus (train) ───────────────────────────────────────────────────────
    print("Carregando corpus TriviaQA (split TRAIN para indexação)...")
    train_data = load_dataset("trivia_qa", "rc.wikipedia", split=CORPUS_SPLIT)

    # ── [M1] Chunking de passagens para retrieval preciso ──────────────
    # Passagens Wikipedia inteiras (~5.000-30.000 chars) geram retrieval
    # impreciso. Chunks de 1024 chars (~256 tokens) com overlap de 256
    # chars permitem retrieval cirúrgico por parágrafo.
    CHUNK_CHARS   = 1024
    OVERLAP_CHARS = 256

    def chunk_passage(text, chunk_chars=CHUNK_CHARS, overlap=OVERLAP_CHARS):
        chunks, start = [], 0
        while start < len(text):
            end   = min(start + chunk_chars, len(text))
            chunk = text[start:end].strip()
            if len(chunk) >= 150:
                chunks.append(chunk)
            if end == len(text):
                break
            start = end - overlap
        return chunks if chunks else [text.strip()]

    raw_passages = []
    for example in train_data:
        for passage in example["entity_pages"]["wiki_context"]:
            if passage and len(passage.strip()) >= MIN_PASSAGE_CHARS:
                raw_passages.append(passage.strip())

    # Deduplicação por hash de prefixo (512 chars)
    seen, dedup = set(), []
    for doc in raw_passages:
        key = doc[:512]
        if key not in seen:
            seen.add(key)
            dedup.append(doc)

    # Chunking com overlap
    documents = []
    for doc in dedup:
        documents.extend(chunk_passage(doc))
    documents = documents[:MAX_CORPUS_PASSAGES]

    print(f"Corpus: {len(documents)} chunks indexados (de {len(dedup)} passagens únicas, split=TRAIN).")
    print(f"Comprimento médio: {sum(len(d) for d in documents) // max(len(documents),1)} chars.")

    # ── Gold Answers (validation) ─────────────────────────────────────────────
    print("Carregando queries de avaliação (split VALIDATION — independente do corpus)...")
    val_data = load_dataset("trivia_qa", "rc.wikipedia", split=EVAL_SPLIT)

    gold_answers = {}
    for example in val_data:
        question = example["question"]
        gold_answers[question] = example["answer"]["aliases"]

    print(f"Queries de avaliação disponíveis: {len(gold_answers)} (split=VALIDATION).")

    with open("gold_answers.json", "w", encoding="utf-8") as f:
        json.dump(gold_answers, f, ensure_ascii=False)

    # ── Indexação ─────────────────────────────────────────────────────────────
    print("Calculando embeddings e escrevendo no DB local...")
    client = chromadb.PersistentClient(path=VECTOR_DB_PATH)
    try:
        client.delete_collection("cognito_knowledge_base")
    except Exception:
        pass

    collection = client.create_collection(
        name="cognito_knowledge_base",
        metadata={"hnsw:space": "cosine"}
    )

    embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)
    embeddings  = embed_model.encode(
        documents,
        batch_size=8,
        normalize_embeddings=True,
        show_progress_bar=True,
    )

    collection.add(
        documents=documents,
        embeddings=embeddings.tolist(),
        ids=[str(i) for i in range(len(documents))],
    )
    print("Fase 1 concluída com sucesso.")
    print(f"  Corpus (TRAIN):      {len(documents)} passagens indexadas.")
    print(f"  Avaliação (VAL):     {len(gold_answers)} queries com gabarito.")
    print("  Data leakage: ZERO — splits são disjuntos por design.")


if __name__ == "__main__":
    main()





In [ ]:
!pip install -q uv
!uv run 1_ingestao.py


## Fase 2: Avaliação Padronizada (LM-Eval)
Roda o pipeline oficial no ambiente limpo e é desmantelado depois.


In [ ]:

%%writefile 2_avaliacao.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "lm-eval>=0.4.4",
#     "transformers>=4.46.0",
#     "accelerate>=0.34.0",
#     "bitsandbytes>=0.44.1",
#     "torch",
#     "einops"
# ]
# ///

import torch
import lm_eval
from lm_eval.models.huggingface import HFLM
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print('Carregando modelo NF4 para avaliação padronizada...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Passar o modelo já instanciado para o HFLM evita conflito de args internos
lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=4)

print('Executando benchmarks padronizados...')
results = lm_eval.simple_evaluate(
    model=lm,
    tasks=['arc_challenge', 'hellaswag', 'winogrande', 'mmlu', 'triviaqa'],
    num_fewshot=0,
    limit=500,
    log_samples=True,
)

print(lm_eval.utils.make_table(results))


In [ ]:
# Execução completa requer ~40-60 min em GPU dedicada (A100/V100).
# Em Colab T4 gratuito o tempo excede o limite de sessão.
# Os resultados abaixo são reportados por referência cruzada —
# Jiang et al. (2023) e HuggingFace Open LLM Leaderboard.
# Quantização NF4 introduz degradação < 2% (Dettmers et al., 2023).
#
# Para reprodução completa, descomente a linha abaixo:
# !uv run 2_avaliacao.py

REFERENCE_RESULTS = {
    "arc_challenge": {"metric": "acc_norm", "score": 0.5998, "source": "HF Open LLM Leaderboard"},
    "hellaswag":     {"metric": "acc_norm", "score": 0.8130, "source": "HF Open LLM Leaderboard"},
    "mmlu":          {"metric": "acc",      "score": 0.6010, "source": "Jiang et al. (2023)"},
    "winogrande":    {"metric": "acc",      "score": 0.7530, "source": "HF Open LLM Leaderboard"},
}

print("Fase 2 — Mistral-7B-Instruct-v0.3 (float16, referência cruzada)")
print(f"\n{'Benchmark':<20} {'Métrica':<12} {'Score':>8}  Fonte")
print("─" * 72)
for task, d in REFERENCE_RESULTS.items():
    print(f"{task:<20} {d['metric']:<12} {d['score']:>8.4f}  {d['source']}")

## Fase 3: RAG Cognito & Avaliação de Paging
Uma máquina com 100% de VRAM dedicada apenas à Inferência Paging Core.


In [ ]:
%%writefile 3_inferencia.py
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "numpy>=1.26.4",
#     "transformers>=4.46.0",
#     "accelerate>=0.34.0",
#     "bitsandbytes>=0.44.1",
#     "peft>=0.12.0",
#     "sentence-transformers>=3.0.0",
#     "chromadb>=0.5.5",
#     "rouge-score>=0.1.2",
#     "scikit-learn>=1.5.1",
#     "torch",
#     "einops"
# ]
# ///

import gc
import numpy as np
import json
import time
import torch
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, CrossEncoder
# rouge_score removido — métricas migradas para EM e F1-Token (TriviaQA oficial)
import warnings
warnings.filterwarnings('ignore')

EMBEDDING_MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
RERANKER_MODEL_NAME  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LLM_MODEL_NAME       = "mistralai/Mistral-7B-Instruct-v0.3"
VECTOR_DB_PATH       = "./chroma_cognito"

MAX_CONTEXT_CHARS  = 40_000
MAX_NEW_TOKENS     = 128
NUM_QUERIES_EVAL   = 50

# Limiar acima do qual o loop token-a-token e ativado (contextos medios/longos).
# Abaixo deste valor, model.generate() nativo e utilizado (fast_path).
PAGING_CONTEXT_THRESHOLD_TOKENS = 1000

# Tamanho de cada chunk no Chunked Prefill (tokens por bloco de entrada).
CHUNKED_PREFILL_CHUNK_SIZE       = 512

# Limiar acima do qual o Chunked Prefill substitui o encode completo.
# Coincide com o threshold de paging para garantir que o pager atue no prefill.
CHUNKED_PREFILL_THRESHOLD_TOKENS = PAGING_CONTEXT_THRESHOLD_TOKENS


def force_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


# -----------------------------------------------------------------------------
# VirtualPageManager -- paging dinamico com limiar fixo
# -----------------------------------------------------------------------------
class VirtualPageManager:
    """
    Gerenciador de paging dinamico de KV Cache em nivel de aplicacao.

    Monitora a VRAM reservada pelo processo CUDA e, ao ultrapassar
    threshold_gb, realiza offloading non-blocking dos blocos historicos
    de past_key_values para a RAM do sistema, reconstruindo um
    DynamicCache valido com _seen_tokens preservado para manter
    a coerencia do RoPE Positional Encoding.
    """

    def __init__(self, threshold_gb: float = 7.5, page_size: int = 1024, sink_size: int = 4):
        self.threshold_bytes  = threshold_gb * (1024 ** 3)
        self.page_size        = page_size
        self.sink_size        = sink_size # StreamingLLM Attention Sinks
        self.cpu_swap_storage = []
        self.active           = False
        self._debug_printed   = False

    def check_vram_pressure(self) -> bool:
        if not torch.cuda.is_available():
            return False
        return torch.cuda.memory_allocated(0) > self.threshold_bytes

    def offload_kv_cache(self, past_key_values):
        if not self.active or past_key_values is None:
            return past_key_values
        if not self.check_vram_pressure():
            return past_key_values

        new_past_kv      = []
        layers_paged     = 0

        # Recupera e congela a extensao original da sequencia logica (RoPE timestamp)
        # ao inves de deduzir do tamanho fisico truncado.
        if hasattr(past_key_values, "seen_tokens"):
            original_seq_len = past_key_values.seen_tokens
        elif hasattr(past_key_values, "get_seq_length"):
            original_seq_len = past_key_values.get_seq_length()
        else:
            original_seq_len = 0

        for layer_idx, layer_cache in enumerate(past_key_values):
            k, v     = layer_cache[0], layer_cache[1]
            seq_len  = k.shape[2]
            if original_seq_len == 0:
                original_seq_len = seq_len

            if seq_len > self.page_size:
                keep_size = self.page_size - self.sink_size
                # Evict middle context. Sink tokens are preserved in VRAM.
                k_cpu = k[:, :, self.sink_size:-keep_size, :].detach().to('cpu', non_blocking=True)
                v_cpu = v[:, :, self.sink_size:-keep_size, :].detach().to('cpu', non_blocking=True)
                self.cpu_swap_storage.append((layer_idx, k_cpu, v_cpu))

                # StreamingLLM: Concatenate Sinks with Recent Context
                k_new = torch.cat([k[:, :, :self.sink_size, :], k[:, :, -keep_size:, :]], dim=2)
                v_new = torch.cat([v[:, :, :self.sink_size, :], v[:, :, -keep_size:, :]], dim=2)
                new_past_kv.append((k_new, v_new))
                layers_paged += 1
            else:
                new_past_kv.append((k, v))

        if layers_paged > 0:
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            gc.collect()
            torch.cuda.empty_cache()

        try:
            from transformers.cache_utils import DynamicCache
            new_cache = DynamicCache()
            keys = [x[0] for x in new_past_kv]
            vals = [x[1] for x in new_past_kv]
            if hasattr(new_cache, "_key_cache"):
                new_cache._key_cache   = keys
                new_cache._value_cache = vals
            else:
                new_cache.key_cache   = keys
                new_cache.value_cache = vals
            if hasattr(new_cache, "_seen_tokens"):
                new_cache._seen_tokens = original_seq_len
            new_cache.seen_tokens = original_seq_len

            if not self._debug_printed:
                print(f"[Cognito Engine] Memory threshold reached. Pager activated.")
                print(f"[Cognito Engine] Input Cache Structure: {type(past_key_values)}")
                print(f"[Cognito Engine] Reconstructed Cache: {type(new_cache)}")
                print(f"[Cognito Engine] Original Positional Encoding: {original_seq_len}")
                print(f"[Cognito Engine] Physical Tensor Size: {new_cache.get_seq_length()}")
                print(f"[Cognito Engine] Offloaded Layers: {layers_paged} camadas paginadas.")
                self._debug_printed = True

            return new_cache
        except Exception as e:
            print(f"[Cognito Engine] Fatal paging exception: {e}")
            raise

    def reset(self):
        self.cpu_swap_storage = []
        self.active           = False
        self._debug_printed   = False
        force_cleanup()

    @property
    def blocks_in_cpu(self) -> int:
        return len(self.cpu_swap_storage)


# -----------------------------------------------------------------------------
# AdaptiveVirtualPageManager -- limiar adaptativo via EMA
# -----------------------------------------------------------------------------
class AdaptiveVirtualPageManager(VirtualPageManager):
    """
    Extensao do VirtualPageManager com politica de limiar adaptativo.

    Atualiza uma EMA do consumo de VRAM observado a cada passo de
    decodificacao e reajusta o threshold como:

        threshold = ema_vram * (1 + safety_margin)

    garantindo intervencao preventiva sem depender de valor fixo manual.
    """

    def __init__(
        self,
        initial_threshold_gb: float = 7.5,
        page_size: int = 1024,
        safety_margin: float = 0.15,
        ema_alpha: float = 0.3,
        warmup_steps: int = 10,
        gpu_capacity_gb: float = 15.5,
    ):
        super().__init__(threshold_gb=initial_threshold_gb, page_size=page_size)
        self.safety_margin   = safety_margin
        self.ema_alpha       = ema_alpha
        self.warmup_steps    = warmup_steps
        self.gpu_capacity_gb = gpu_capacity_gb
        self._ema_gb         = None
        self._step           = 0

    def _update_threshold(self):
        if not torch.cuda.is_available():
            return
        # [F0.1] memory_allocated() mede uso real dos tensores;
        # memory_reserved() inclui o pool do driver CUDA e não reflete
        # o consumo do modelo. Uso de allocated() garante consistência
        # com as métricas reportadas na tabela de ablação.
        current_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
        self._step += 1
        if self._ema_gb is None:
            self._ema_gb = current_gb
        else:
            self._ema_gb = self.ema_alpha * current_gb + (1 - self.ema_alpha) * self._ema_gb
        if self._step > self.warmup_steps:
            adaptive_gb = min(
                self._ema_gb * (1 + self.safety_margin),
                self.gpu_capacity_gb * 0.92,
            )
            self.threshold_bytes = adaptive_gb * (1024 ** 3)

    def offload_kv_cache(self, past_key_values):
        self._update_threshold()
        return super().offload_kv_cache(past_key_values)

    def reset(self):
        super().reset()
        self._ema_gb = None
        self._step   = 0


# -----------------------------------------------------------------------------

# -----------------------------------------------------------------------------
# PredictiveMemoryPolicy -- limiar preditivo baseado em contexto
# -----------------------------------------------------------------------------
class PredictiveMemoryPolicy(AdaptiveVirtualPageManager):
    """
    [F0.2] Política Preditiva de Memória — implementação causal validada.

    Estima geometricamente o incremento de VRAM provocado pelo KV Cache
    ANTES que o passo de decodificação ocorra. O offload é acionado se:

        allocated_now + kv_increment_estimate > threshold

    Isso é distinto da política reativa (AdaptiveVirtualPageManager), que
    consulta memory_allocated() APÓS o passo, quando o pico já ocorreu.

    Parâmetros do Mistral-7B (GQA):
      - num_kv_heads = 8 (Grouped Query Attention)
      - head_dim     = 128
      - num_layers   = 32
      - dtype        = float16 (2 bytes/elemento)
      - K + V        = fator 2

    KV_cost(Δlen) = Δlen * 8 * 128 * 32 * 2 * 2 bytes
                  = Δlen * 131_072 bytes
    """

    # Constantes arquiteturais do Mistral-7B-Instruct-v0.3
    NUM_KV_HEADS    = 8
    HEAD_DIM        = 128
    NUM_LAYERS      = 32
    BYTES_PER_ELEM  = 2  # float16
    KV_FACTOR       = 2  # K and V tensors

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def predict_kv_increment_gb(self, delta_tokens: int) -> float:
        """Retorna o incremento estimado do KV cache em GB para delta_tokens novos tokens."""
        bytes_cost = (
            delta_tokens
            * self.NUM_KV_HEADS
            * self.HEAD_DIM
            * self.NUM_LAYERS
            * self.KV_FACTOR
            * self.BYTES_PER_ELEM
        )
        return bytes_cost / (1024 ** 3)

    def should_offload_predictive(self, delta_tokens: int = 1) -> bool:
        """Decide preditivamente se offload é necessário antes do próximo passo."""
        if not torch.cuda.is_available():
            return False
        current_gb  = torch.cuda.memory_allocated(0) / (1024 ** 3)
        predict_gb  = self.predict_kv_increment_gb(delta_tokens)
        threshold_gb = self.threshold_bytes / (1024 ** 3)
        return (current_gb + predict_gb) > threshold_gb

    def auto_calibrate(self, model_baseline_gb: float, headroom_gb: float = 0.9):
        """Calibra threshold relativo ao footprint real do modelo."""
        threshold_gb = model_baseline_gb + headroom_gb
        self.threshold_bytes = threshold_gb * (1024 ** 3)
        print(f"[PredictivePolicy] Threshold auto-calibrado: {threshold_gb:.2f} GB")

    def offload_kv_cache(self, past_key_values):
        """Override: usa decisão preditiva em vez de verificação reativa."""
        if not self.active or past_key_values is None:
            return past_key_values
        # Usa estimativa preditiva de 1 token futuro para disparar offload
        if not self.should_offload_predictive(delta_tokens=1):
            return past_key_values
        # Aplica o offload físico da classe pai (que já tem a lógica de StreamingLLM Sinks)
        self.active = True  # garante que super() processe
        # Chamamos diretamente VirtualPageManager.offload_kv_cache para evitar
        # a verificação reativa do Adaptive e preservar o comportamento preditivo.
        return VirtualPageManager.offload_kv_cache(self, past_key_values)


# =============================================================================
# RAGAwarePager -- evicção por relevância para RAG (contraste com StreamingLLM)
# =============================================================================
from dataclasses import dataclass, field as dc_field

class RAGAwarePager(PredictiveMemoryPolicy):
    """
    Pager com política de evicção por relevância semântica.

    Motivação: StreamingLLM Sinks (Xiao et al., 2023) pressupõe localidade
    TEMPORAL — tokens recentes são mais importantes. Em RAG, a relevância é
    determinada pelo score do reranker, não pela posição da passagem no prompt.
    Evictar por posição descarta passagens relevantes; evictar por score
    descarta passagens que o retriever já classificou como menos úteis.

    Protocolo de evicção:
      1. Cada passagem recuperada é registrada como um KVSegment com
         seu score de reranking e posição no KV cache.
      2. Ao ultrapassar o threshold preditivo, o segmento de MENOR score
         é removido fisicamente do KV cache de todas as camadas.
      3. As posições dos segmentos remanescentes são atualizadas.
      4. Para multi-turno: score = índice do turno (turno mais antigo = menor).

    Referência de contraste: Xiao et al. (2023). Efficient Streaming Language
    Models with Attention Sinks. ICLR 2024.
    """

    @dataclass
    class KVSegment:
        seg_id:     int
        score:      float   # reranker score (RAG) ou recency score (diálogo)
        start_pos:  int     # posição inicial no KV físico
        end_pos:    int     # posição final (exclusive)
        label:      str = ""
        turn_id:    int = 0

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._segments:     list  = []
        self._next_seg_id:  int   = 0
        self._current_pos:  int   = 0
        self._evictions:    int   = 0
        self.global_turn:   int   = 0

    # ── Calibração ─────────────────────────────────────────────────────────
    def auto_calibrate(self, model_baseline_gb: float, headroom_gb: float = 0.9):
        """
        Define threshold como baseline_modelo + headroom.

        Sem auto-calibração, o threshold de 7.5 GB (calibrado para modelos
        float16 com ~14 GB de pesos) nunca é atingido em configurações NF4
        (~4 GB de pesos), tornando o pager inativo.

        Com NF4 (baseline ~4 GB) + headroom 0.9 GB → threshold = 4.9 GB.
        O pager dispara ao acumular ~900 MB de KV cache, o que ocorre com
        ~7.000 tokens de contexto acumulado.
        """
        threshold_gb = model_baseline_gb + headroom_gb
        self.threshold_bytes = threshold_gb * (1024 ** 3)
        print(
            f"[RAGAwarePager] Auto-threshold: "
            f"{model_baseline_gb:.2f} GB (modelo) + {headroom_gb:.2f} GB (headroom) "
            f"= {threshold_gb:.2f} GB"
        )

    # ── Registro de segmentos ──────────────────────────────────────────────
    def register_segment(self, token_count: int, score: float, label: str = "", turn_id: int = -1) -> int:
        """Registra passagem/turno como segmento KV endereçável."""
        seg = RAGAwarePager.KVSegment(
            seg_id    = self._next_seg_id,
            score     = score,
            start_pos = self._current_pos,
            end_pos   = self._current_pos + token_count,
            label     = label,
            turn_id   = self.global_turn if turn_id == -1 else turn_id,
        )
        self._segments.append(seg)
        self._current_pos += token_count
        self._next_seg_id += 1
        return seg.seg_id

    # ── Evicção por score ──────────────────────────────────────────────────
    def offload_kv_cache(self, past_key_values):
        """Override: evicta segmento de menor score, não sinks temporais."""
        if not self.active or past_key_values is None:
            return past_key_values
        if not self.should_offload_predictive(delta_tokens=1):
            return past_key_values
        if not self._segments:
            # Fallback: sem segmentos registrados, usa evicção da classe pai
            return VirtualPageManager.offload_kv_cache(self, past_key_values)

        # Seleciona vítima: menor score de reranking
        import math
        def ebbinghaus_score(s):
            lag = self.global_turn - getattr(s, "turn_id", self.global_turn)
            return s.score * math.exp(-0.4 * lag)
        self._segments.sort(key=ebbinghaus_score)
        victim = self._segments.pop(0)

        return self._evict_segment(past_key_values, victim)

    def _evict_segment(self, past_key_values, victim):
        """Remove tokens do segmento vítima do KV cache de todas as camadas."""
        from transformers.cache_utils import DynamicCache

        start, end = victim.start_pos, victim.end_pos

        new_cache            = DynamicCache()
        new_cache.key_cache   = []
        new_cache.value_cache = []

        for layer_idx in range(len(past_key_values.key_cache)):
            k   = past_key_values.key_cache[layer_idx]   # [1, heads, seq, dim]
            v   = past_key_values.value_cache[layer_idx]
            seq = k.shape[2]

            s = min(start, seq)
            e = min(end,   seq)

            if s == e:                                   # segmento vazio após clamp
                new_cache.key_cache.append(k)
                new_cache.value_cache.append(v)
                continue

            if s == 0:
                k_kept = k[:, :, e:,  :]
                v_kept = v[:, :, e:,  :]
            elif e >= seq:
                k_kept = k[:, :, :s, :]
                v_kept = v[:, :, :s, :]
            else:
                k_kept = torch.cat([k[:, :, :s, :], k[:, :, e:, :]], dim=2)
                v_kept = torch.cat([v[:, :, :s, :], v[:, :, e:, :]], dim=2)

            new_cache.key_cache.append(k_kept.contiguous())
            new_cache.value_cache.append(v_kept.contiguous())

        # Ajusta posições dos segmentos remanescentes
        evicted_len = min(end, past_key_values.key_cache[0].shape[2]) - min(start, past_key_values.key_cache[0].shape[2])
        for seg in self._segments:
            if seg.start_pos >= end:
                seg.start_pos -= evicted_len
                seg.end_pos   -= evicted_len
            elif seg.end_pos > start:
                # Sobreposição parcial: ajusta fim
                seg.end_pos = max(seg.start_pos, seg.end_pos - evicted_len)

        self._current_pos -= evicted_len
        self._evictions   += 1

        # Atualiza seen_tokens pelo tamanho físico real (sem "ghost" positions)
        phys_len = new_cache.key_cache[0].shape[2] if new_cache.key_cache else 0
        new_cache.seen_tokens = phys_len
        if hasattr(new_cache, "_seen_tokens"):
            new_cache._seen_tokens = phys_len

        if not self._debug_printed:
            print(
                f"[RAGAwarePager] Evicção #{self._evictions}: "
                f"segmento '{victim.label}' (score={victim.score:.4f}, "
                f"{evicted_len} tokens, pos {victim.start_pos}-{victim.end_pos})"
            )
            self._debug_printed = True

        # Libera tensores
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

        return new_cache

    def reset(self):
        super().reset()
        self._segments.clear()
        self._next_seg_id = 0
        self._current_pos = 0
        self._evictions   = 0

    @property
    def eviction_count(self) -> int:
        return self._evictions

# -----------------------------------------------------------------------------
# HardenedLLMEngine
# -----------------------------------------------------------------------------
class HardenedLLMEngine:
    def __init__(
        self,
        model_name: str,
        vector_db_path: str,
        paging_manager=None,
        top_k_retrieval: int = 10,
        top_k_rerank: int = 3,
    ):
        self.model_name      = model_name
        self.vector_db_path  = vector_db_path
        self.pager           = paging_manager
        self.top_k_retrieval = top_k_retrieval
        self.top_k_rerank    = top_k_rerank
        self.model           = None
        self.tokenizer       = None
        self.collection      = None
        self._embed_model    = None
        self._rerank_model   = None
        self.cached_prefix_kv = None
        self.cached_prefix_len = 0
        # [M2] Prompt otimizado para respostas curtas e diretas.
        self.system_prefix = (
            "[INST] You are a precise factual assistant. "
            "Answer the question using ONLY the provided context. "
            "Give a SHORT, DIRECT answer in 1 to 5 words. "
            "Do not explain or elaborate.\n\n"
        )

    def initialize_runtime(self):
        import torch
        if torch.cuda.is_available():
            torch.cuda.set_per_process_memory_fraction(0.35, device=0)
            print("[Emulador TCC] Alocacao PyTorch restrita a 35% de uma T4 (~5.4GB totais).")
        client          = chromadb.PersistentClient(path=self.vector_db_path)
        self.collection = client.get_collection("cognito_knowledge_base")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # PyTorch SDPA (Scaled Dot Product Attention) -- implementacao nativa
        # do PyTorch 2.0+ que utiliza kernels de atencao eficientes em memoria.
        # Aplicada a TODAS as configuracoes (baseline e Cognito) para garantir
        # comparacao metodologicamente justa. Sem dependencia externa alem do
        # PyTorch, preservando a premissa de operacao na camada de aplicacao.
        try:
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=bnb_config,
                device_map="auto",
                torch_dtype=torch.float16,
                attn_implementation="sdpa",
            )
            print("[Cognito Engine] attn_implementation=sdpa ativo.")
        except Exception:
            print("[Cognito Engine] SDPA indisponivel, usando implementacao padrao.")
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=bnb_config,
                device_map="auto",
                torch_dtype=torch.float16,
            )
        self.model.eval()

        # [P7] Auto-calibra threshold do pager relativo ao footprint NF4 real
        if self.pager is not None and hasattr(self.pager, 'auto_calibrate'):
            baseline_gb = torch.cuda.memory_allocated(0) / (1024 ** 3)
            self.pager.auto_calibrate(baseline_gb, headroom_gb=0.9)

        self._embed_model  = SentenceTransformer(EMBEDDING_MODEL_NAME, trust_remote_code=True)
        self._rerank_model = CrossEncoder(RERANKER_MODEL_NAME)

        # Precompute Automatic Prefix Caching (APC)
        print("[Cognito Engine] Compilando Prefix Cache RAG...")
        device = next(self.model.parameters()).device
        prefix_enc = self.tokenizer(self.system_prefix, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = self.model(input_ids=prefix_enc["input_ids"], attention_mask=prefix_enc["attention_mask"], use_cache=True)
            self.cached_prefix_kv = outputs.past_key_values
            self.cached_prefix_len = prefix_enc["input_ids"].shape[-1]
            print(f"[Cognito Engine] Prefix Cache gerado: {self.cached_prefix_len} tokens retidos em MvRAM quente.")

    def retrieve(self, query: str) -> list:
        """Retorna documentos (compatibilidade retroativa)."""
        return [doc for doc, _ in self.retrieve_with_scores(query)]

    def retrieve_with_scores(self, query: str) -> list:
        """
        Retorna [(doc, reranker_score)] ordenado por score descendente.
        Os scores são usados pelo RAGAwarePager para evicção por relevância.
        """
        query_vec  = self._embed_model.encode([query], normalize_embeddings=True)
        results    = self.collection.query(
            query_embeddings=query_vec.tolist(),
            n_results=min(self.top_k_retrieval, self.collection.count()),
        )
        candidates = results["documents"][0]
        if not candidates:
            return []
        pairs  = [(query, doc) for doc in candidates]
        scores = self._rerank_model.predict(pairs)
        ranked = sorted(zip(scores, candidates), reverse=True)
        return [(doc, float(sc)) for sc, doc in ranked[: self.top_k_rerank]]


    def _passage_level_prefill(
        self,
        docs_scores: list,
        attention_mask_prefix: torch.Tensor,
        initial_past_kv=None,
    ):
        """
        Processa cada passagem individualmente, registrando-a como
        segmento KV endereçável no RAGAwarePager.

        Permite que o pager evicte a passagem MENOS relevante (menor score
        do reranker) ao invés de aplicar critério temporal (StreamingLLM).

        Parâmetros
        ----------
        docs_scores : list[(str, float)]
            Passagens recuperadas com seus scores de reranking.
        attention_mask_prefix : Tensor [1, prefix_len]
            Máscara do prefix cache já computado.
        initial_past_kv : DynamicCache | None
            KV cache do prefix (resultado do APC).
        """
        device  = next(self.model.parameters()).device
        past_kv = initial_past_kv

        if isinstance(self.pager, RAGAwarePager):
            self.pager.reset()
            self.pager.active = True

        # [BUG-5 FIX] Força "Context:\n" no início mesmo com prefix
        if past_kv is None:
            prefix_str = self.system_prefix + "Context:\n"
            add_special = True
        else:
            prefix_str = "Context:\n"
            add_special = False
            
        prefix_enc = self.tokenizer(
            prefix_str, return_tensors="pt", add_special_tokens=add_special
        )
        prefix_ids = prefix_enc["input_ids"].to(device)
        kv_len = past_kv.seen_tokens if (past_kv and hasattr(past_kv, "seen_tokens")) else 0
        prefix_mask = torch.ones((1, kv_len + prefix_ids.shape[-1]), dtype=torch.long, device=device)
        
        with torch.no_grad():
            prefix_out = self.model(
                input_ids=prefix_ids,
                attention_mask=prefix_mask,
                past_key_values=past_kv,
                use_cache=True,
            )
        past_kv = prefix_out.past_key_values

        for i, (passage, score) in enumerate(docs_scores):
            passage_text = passage + "\n" if i < len(docs_scores) - 1 else passage
            passage_enc = self.tokenizer(
                passage_text, return_tensors="pt", add_special_tokens=False
            )
            passage_ids = passage_enc["input_ids"].to(device)
            n_tokens    = passage_ids.shape[-1]

            # Máscara: cobre prefix + todos os segmentos processados até agora
            kv_len     = past_kv.seen_tokens if (past_kv and hasattr(past_kv, "seen_tokens")) else 0
            total_len  = kv_len + n_tokens
            chunk_mask = torch.ones((1, total_len), dtype=torch.long, device=device)

            with torch.no_grad():
                out     = self.model(
                    input_ids      = passage_ids,
                    attention_mask = chunk_mask,
                    past_key_values= past_kv,
                    use_cache      = True,
                )
            past_kv = out.past_key_values

            # Registra segmento e decide evicção (score-based)
            if isinstance(self.pager, RAGAwarePager):
                self.pager.register_segment(
                    token_count = n_tokens,
                    score       = score,
                    label       = f"passage_{i}_score{score:.3f}",
                    turn_id     = getattr(self.pager, 'global_turn', 0),
                )
                past_kv = self.pager.offload_kv_cache(past_kv)

# VirtualPageManager -- paging dinamico com limiar fixo


In [ ]:
!uv run 3_inferencia.py


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CÉLULA DE FIM DE SESSÃO — Execute ANTES de fechar o Colab
# ═══════════════════════════════════════════════════════════════
# Persiste dados locais de volta ao Drive.
# IMPORTANTE: rodar antes de encerrar a sessão ou antes do timeout.

import os, shutil, time, json as _j

DRIVE_ROOT = '/content/drive/MyDrive/cognito_tcc'
LOCAL_ROOT = '/content'

os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)

saved = []

# ── Salva ChromaDB ────────────────────────────────────────────────
LOCAL_CHROMA = f'{LOCAL_ROOT}/chroma_cognito'
DRIVE_CHROMA = f'{DRIVE_ROOT}/chroma_cognito'
if os.path.exists(f'{LOCAL_CHROMA}/chroma.sqlite3'):
    print('Salvando ChromaDB → Drive ...')
    t0 = time.time()
    if os.path.exists(DRIVE_CHROMA):
        shutil.rmtree(DRIVE_CHROMA)
    shutil.copytree(LOCAL_CHROMA, DRIVE_CHROMA)
    size_mb = sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(DRIVE_CHROMA) for f in fs) / 1e6
    print(f'  ✓ ChromaDB salvo: {size_mb:.1f} MB em {time.time()-t0:.1f}s')
    saved.append('chroma_cognito')
else:
    print('  ⚠ ChromaDB não encontrado em /content/ — nada a salvar.')

# ── Salva gold_answers.json ───────────────────────────────────────
LOCAL_GOLD = f'{LOCAL_ROOT}/gold_answers.json'
if os.path.exists(LOCAL_GOLD):
    shutil.copy(LOCAL_GOLD, f'{DRIVE_ROOT}/gold_answers.json')
    print(f'  ✓ gold_answers.json salvo.')
    saved.append('gold_answers.json')

# ── Salva checkpoints JSONL ───────────────────────────────────────
ckpt_files = [f for f in os.listdir(LOCAL_ROOT)
              if f.startswith('results_checkpoint_') and f.endswith('.jsonl')]
for ck in ckpt_files:
    src = f'{LOCAL_ROOT}/{ck}'
    dst = f'{DRIVE_ROOT}/checkpoints/{ck}'
    shutil.copy(src, dst)
    # Conta linhas (queries completadas)
    n_lines = sum(1 for _ in open(src))
    print(f'  ✓ {ck}: {n_lines} queries salvas.')
    saved.append(ck)

# ── Relatório ─────────────────────────────────────────────────────
print()
if saved:
    print(f'✓ Sessão salva no Drive: {saved}')
    print(f'  Caminho: {DRIVE_ROOT}')
else:
    print('⚠ Nada foi salvo — verifique os caminhos.')

# ── Status dos checkpoints (progresso da ablação) ─────────────────
print()
print('Status da Ablação:')
all_ckpts = [f for f in os.listdir(f'{DRIVE_ROOT}/checkpoints') if f.endswith('.jsonl')]
if all_ckpts:
    for ck in sorted(all_ckpts):
        path = f'{DRIVE_ROOT}/checkpoints/{ck}'
        lines = open(path).readlines()
        ooms = sum(1 for l in lines if '"OOM_FAILURE"' in l)
        ok = len(lines) - ooms
        print(f'  {ck:<45} {ok:>3} OK  {ooms:>3} OOM  ({len(lines):>3} total)')
else:
    print('  Nenhum checkpoint encontrado ainda.')
